# PROVIDER - Silver Layer

This notebook validates the MIMIC-IV `provider` Bronze table before storing it in the Silver layer.

`PROVIDER` is a reference table containing the de-identified provider IDs used in hospital order data such as `POE`. It does not contain names or provider details, so its main purpose is to provide a valid reference for `provider_id`.

The Silver check focuses on confirming that `provider_id` remains complete and unique.

In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

provider = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.provider_raw")

total = provider.count()
unique_ids = provider.select("provider_id").distinct().count()
null_ids = provider.filter(F.col("provider_id").isNull()).count()

print(f"Rows: {total:,}")
print(f"Unique provider_id: {unique_ids:,}")
print(f"Null provider_id: {null_ids:,}")
print(f"Duplicate provider_id: {total - unique_ids:,}")

### Silver Result

`PROVIDER` is already clean at the identifier level, with no missing or duplicate `provider_id` values.

Because the table only contains the de-identified provider identifier, no additional cleaning is required before storing it in Silver.

In [0]:
# Save PROVIDER Silver table
(
    provider.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.provider")
)

print("Created: workspace.silver.provider")

In [0]:
bronze_count = spark.table("workspace.bronze.provider_raw").count()
silver_count = spark.table("workspace.silver.provider").count()

print(f"Bronze: {bronze_count:,}")
print(f"Silver: {silver_count:,}")
print(f"Match: {bronze_count == silver_count}")